---

## Module 10 — Persistence in LangGraph

> *"Persistence is a foundational topic in LangGraph — many advanced features are built on top of it."*

---

### 📖 Definition

> **Persistence** in LangGraph refers to the ability to **save and restore the state of a workflow over time.**

---

### 🔁 LangGraph's Default Behavior (without persistence)
```
invoke() → nodes execute → state updates → END → ❌ state erased from RAM
```

Once a workflow reaches END, all state values are gone. Future invocations start from scratch — the workflow has no memory of what happened before.

---

### ✅ With Persistence
```
invoke() → nodes execute → state updates → END → ✅ state saved
invoke() → state restored from storage → continues with full history
```

You can save the state and restore it any time in the future.

---

### 🔑 The Most Important Detail — Intermediate States

Persistence doesn't just save the **final** state. It saves the state at **every intermediate step**:
```
START  → name = "A"   ← saved ✅
Node 1 → name = "B"   ← saved ✅
Node 2 → name = "C"   ← saved ✅
END    → name = "C"   ← saved ✅
```

This is why the definition says *"over time"* — it's a full timeline of state snapshots, not just the end result.

---

### 💡 Why Does This Matter? — Fault Tolerance

Because every intermediate state is saved, if your workflow crashes mid-execution (server goes down, API fails, network error), you don't have to restart from the beginning:
```
Node 1 ✅ saved
Node 2 💥 crash

→ Fix the issue
→ Re-trigger the workflow
→ Resumes from Node 2, not from START
```

> 📌 *This is what **Fault Tolerance** means in LangGraph — and it comes entirely from persistence. Without persistence, there's no fault tolerance.*

---

### 🧱 Persistence is the Foundation For

| Feature | How persistence enables it |
|---|---|
| **Memory** | Restores conversation history across sessions |
| **Fault Tolerance** | Resume from crash point, not from scratch |
| **Human-in-the-Loop** | Pause workflow, wait for human input, resume |
| **Retry Logic** | Re-run a failed node with saved context |
| **Multi-session Chatbots** | Each user's state persisted independently |

> 📌 *All of these features are built on top of persistence. This is why it's called a foundational concept.*

---



---

### 🔍 Persistence — Deep Dive: Checkpointers & Threads

---

#### Where does state get saved?

Some sort of **database**. LangGraph saves state values to a database at every checkpoint, so you can retrieve them any time in the future.

---

### ⚙️ Checkpointers

Persistence in LangGraph is implemented via a **Checkpointer**. It divides your graph execution into **checkpoints** and saves state at each one.

**When does a checkpoint occur?**
At every **superstep** — one superstep = one round of node execution (parallel nodes in the same round count as one superstep).
```
Graph with 2 sequential nodes:

START → Node 1 → Node 2 → END
   ↑         ↑        ↑       ↑
  CP 0     CP 1    CP 2    CP 3
```
```
Graph with parallel nodes:

START → Node 1 → [Node 2, Node 3, Node 4] → END
   ↑         ↑              ↑                  ↑
  CP 0     CP 1           CP 2              CP 3
```

> 📌 *Parallel nodes executing in the same round = one superstep = one checkpoint. So a 5-node parallel graph still only creates 4 checkpoints total, not 7.*

**What gets saved at each checkpoint?**

The full state at that point in time. Example with `numbers: Annotated[list[int], operator.add]`:
```
Checkpoint 0 (START):     numbers = [1]
Checkpoint 1 (Node 1):    numbers = [1, 2]
Checkpoint 2 (Nodes 2-4): numbers = [1, 2, 3, 4, 5]
Checkpoint 3 (END):       numbers = [1, 2, 3, 4, 5]
```

All four snapshots are saved to the database — not just the final one.

---

### 🧵 Threads

Every time you invoke a workflow with persistence enabled, you assign it a **thread ID**. This is how LangGraph knows which saved states belong to which execution.
```
Execution 1: thread_id = 1
  → saves checkpoints [1], [1,2], [1,2,3,4,5] under thread_id=1

Execution 2: thread_id = 2
  → saves checkpoints [6], [6,7], [6,7,8,9,10] under thread_id=2
```

To retrieve a specific execution's state later:
```python
# "give me everything saved under thread_id=2"
config = {"configurable": {"thread_id": 2}}
state = workflow.get_state(config)
```

**Real-world chatbot example:**
```
User starts new chat    → create thread_id=1, save messages to DB under id=1
User starts another     → create thread_id=2, save messages to DB under id=2
User wants to resume    → fetch thread_id=1 from DB → full history restored ✅
```

> 📌 *This is exactly how ChatGPT's "resume past conversation" feature works — every conversation session has a unique thread ID, and all messages are persisted against it.*

---

### 🗂️ Summary

| Concept | What it does |
|---|---|
| **Persistence** | Saves and restores workflow state over time |
| **Checkpointer** | The component that handles saving — triggers at every superstep |
| **Checkpoint** | A snapshot of state at one superstep |
| **Thread ID** | Unique key that groups all checkpoints from one execution |
| **Database** | Where checkpoints are actually stored (RAM for dev, DB for prod) |

---

---

### 💾 Implementing Persistence — Joke Generator Workflow

A simple 2-node sequential workflow used to demonstrate all persistence concepts in code.

---

#### Setup

In [6]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langchain_openai import ChatOpenAI
from typing import TypedDict
from dotenv import load_dotenv

load_dotenv(override =True)
llm = ChatOpenAI()

> 📌 *`InMemorySaver` is LangGraph's built-in RAM-based checkpointer — perfect for demos and learning. For production, swap with a database-backed checkpointer like `PostgresSaver` or `RedisSaver`. The API is identical — only the import changes.*

---

#### State

In [7]:
class JokeState(TypedDict):
    topic: str        # user-provided topic
    joke: str         # generated by Node 1
    explanation: str  # generated by Node 2

---

#### Nodes

In [8]:
def generate_joke(state: JokeState):
    """Generate a joke on the given topic."""
    prompt = f'Generate a joke on the topic {state["topic"]}'
    response = llm.invoke(prompt).content
    return {'joke': response}


def generate_explanation(state: JokeState):
    """Explain why the joke is funny."""
    prompt = f'Write an explanation for the joke - {state["joke"]}'
    response = llm.invoke(prompt).content
    return {'explanation': response}

---

#### Graph + Checkpointer

In [9]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

# passing checkpointer here activates persistence for this workflow
workflow = graph.compile(checkpointer=checkpointer)

---

#### Run with a thread ID

In [10]:
# thread_id groups all checkpoints from this execution together
config1 = {"configurable": {"thread_id": "1"}}

workflow.invoke({'topic': 'pizza'}, config=config1)
# → joke: "Why did the pizza go to the doctor? Because it was feeling a little cheesy."
# → explanation: "The joke plays on the double meaning of 'cheesy'..."


{'topic': 'pizza',
 'joke': 'Why was the pizza maker always calm? Because he knew he could always "dough" it!',
 'explanation': 'This joke plays on the wordplay between "dough" as in the dough used to make pizza crust and "do it." The pizza maker is always calm because he knows he can always "dough" it, referring to his ability to make pizza crust, but also implying that he can handle any situation that comes his way. The humor comes from the double meaning of "dough" in this context.'}

> 📌 *Every invocation that shares the same `thread_id` is treated as one continuous session. Different `thread_id` = completely separate state history.*

---

#### Fetch final state

In [11]:
# retrieve the saved final state for thread "1"
workflow.get_state(config1)

# output:
# StateSnapshot(
#   values = {'topic': 'pizza', 'joke': '...', 'explanation': '...'},
#   next = ()   ← empty means workflow has ended
# )

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why was the pizza maker always calm? Because he knew he could always "dough" it!', 'explanation': 'This joke plays on the wordplay between "dough" as in the dough used to make pizza crust and "do it." The pizza maker is always calm because he knows he can always "dough" it, referring to his ability to make pizza crust, but also implying that he can handle any situation that comes his way. The humor comes from the double meaning of "dough" in this context.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1238ba-251c-68ba-8002-65e178f9be54'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-03-19T12:03:25.589087+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1238ba-0f07-6ebc-8001-eef5c6626737'}}, tasks=(), interrupts=())

> 📌 *`next` tells you which node would execute next. Empty tuple = workflow is at END. This field becomes important for Human-in-the-Loop — you can pause a workflow mid-execution and `next` shows exactly where it stopped.*

---

#### Fetch full state history (all checkpoints)

In [12]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why was the pizza maker always calm? Because he knew he could always "dough" it!', 'explanation': 'This joke plays on the wordplay between "dough" as in the dough used to make pizza crust and "do it." The pizza maker is always calm because he knows he can always "dough" it, referring to his ability to make pizza crust, but also implying that he can handle any situation that comes his way. The humor comes from the double meaning of "dough" in this context.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1238ba-251c-68ba-8002-65e178f9be54'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-03-19T12:03:25.589087+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1238ba-0f07-6ebc-8001-eef5c6626737'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'topic': 'pizza', 'joke': 'Why was the pizza maker always calm? Because he

Returns **4 snapshots** — one per superstep (one per checkpoint):
```
Checkpoint 4 — just before END
  values: {topic: "pizza", joke: "...", explanation: "..."}
  next: ()

Checkpoint 3 — just before generate_explanation
  values: {topic: "pizza", joke: "...", explanation: None}
  next: ("generate_explanation",)

Checkpoint 2 — just before generate_joke
  values: {topic: "pizza", joke: None, explanation: None}
  next: ("generate_joke",)

Checkpoint 1 — just before START
  values: {topic: None, joke: None, explanation: None}
  next: ("__start__",)
```

> 📌 *State history is returned in **reverse chronological order** — most recent checkpoint first. Each snapshot also tells you which node was about to execute next, giving you a complete audit trail of the entire workflow run.*

---

#### Checkpointer Options

| Checkpointer | Storage | Use case |
|---|---|---|
| `InMemorySaver` | RAM | Learning, demos |
| `PostgresSaver` | PostgreSQL | Production |
| `RedisSaver` | Redis | Production (high throughput) |

All checkpointers share the same API — only the import and instantiation differ.

---



---

### 🧵 Multiple Threads — Isolated State per Execution

Each invocation with a different `thread_id` gets its own completely isolated state history in the database.

---

In [13]:
# second execution — different topic, different thread
config2 = {"configurable": {"thread_id": "2"}}
workflow.invoke({'topic': 'pasta'}, config=config2)

# fetch final state for thread 2
workflow.get_state(config2)

StateSnapshot(values={'topic': 'pasta', 'joke': "Why did the spaghetti go to the party alone? Because it couldn't find a date!", 'explanation': "This joke plays on the fact that spaghetti is an inanimate object and therefore cannot actually go to a party, let alone bring a date. The humor comes from the idea of spaghetti having human-like qualities such as the ability to find a date for a party. The punchline reveals the absurdity of the situation by pointing out that spaghetti wouldn't need a date in the first place because it is just a food item."}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f123933-9a90-66ea-8002-0078befa5f5e'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-03-19T12:57:45.973919+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f123933-885e-61a0-8001-7f850df6f2dc'}}, tasks=(), interrupts=())

In [14]:
# fetch full checkpoint history for thread 2
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'pasta', 'joke': "Why did the spaghetti go to the party alone? Because it couldn't find a date!", 'explanation': "This joke plays on the fact that spaghetti is an inanimate object and therefore cannot actually go to a party, let alone bring a date. The humor comes from the idea of spaghetti having human-like qualities such as the ability to find a date for a party. The punchline reveals the absurdity of the situation by pointing out that spaghetti wouldn't need a date in the first place because it is just a food item."}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f123933-9a90-66ea-8002-0078befa5f5e'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-03-19T12:57:45.973919+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f123933-885e-61a0-8001-7f850df6f2dc'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'topic': 'pasta', 'joke': "W

---

#### Thread isolation in action

```
workflow.get_state(config1)  → pizza joke + explanation  ✅
workflow.get_state(config2)  → pasta joke + explanation  ✅

workflow.get_state_history(config1)  → 4 checkpoints for pizza run
workflow.get_state_history(config2)  → 4 checkpoints for pasta run
```

Both are independently stored and retrievable at any future point — as long as the checkpointer (or database) persists.

> 📌 *This is the direct equivalent of ChatGPT's conversation list — each `thread_id` is one conversation, fully isolated from all others. Fetching by `thread_id` = resuming that exact conversation.*

---

#### Persistence summary
```
Without persistence:  invoke() → run → END → state erased
With persistence:     invoke() → run → END → state saved per thread_id
                      invoke() again → state restored → continues from where it left off
```

| Method | What it returns |
|---|---|
| `workflow.get_state(config)` | Final state snapshot for that thread |
| `workflow.get_state_history(config)` | All intermediate + final snapshots for that thread |

---


---

### ⚡ Fault Tolerance via Persistence

If a workflow crashes mid-execution, persistence lets you resume from the exact crash point — not from the beginning.

---

#### State

In [15]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict
import time

class CrashState(TypedDict):
    input: str
    step1: str
    step2: str
    step3: str

---

#### Nodes

In [16]:
def step_1(state: CrashState):
    """Execute step 1 and mark it done."""
    print("✅ Step 1 executed")
    return {"step1": "done"}

def step_2(state: CrashState):
    """Simulate a long-running / hanging node — interrupt this to simulate a crash."""
    print("⏳ Step 2 running... interrupt now to simulate crash")
    time.sleep(30)  # manually interrupt during this sleep
    return {"step2": "done"}

def step_3(state: CrashState):
    """Execute step 3 and mark it done."""
    print("✅ Step 3 executed")
    return {"step3": "done"}

---

#### Graph

In [17]:
builder = StateGraph(CrashState)

builder.add_node("step_1", step_1)
builder.add_node("step_2", step_2)
builder.add_node("step_3", step_3)

builder.add_edge(START, "step_1")
builder.add_edge("step_1", "step_2")
builder.add_edge("step_2", "step_3")
builder.add_edge("step_3", END)

checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)


---

#### Step 1 — Run and crash during step_2

In [18]:
config = {"configurable": {"thread_id": "thread-1"}}

try:
    graph.invoke({"input": "start"}, config=config)
except KeyboardInterrupt:
    print("❌ Crash simulated — workflow interrupted during step_2")

✅ Step 1 executed
⏳ Step 2 running... interrupt now to simulate crash
❌ Crash simulated — workflow interrupted during step_2


```
✅ Step 1 executed
⏳ Step 2 running... interrupt now to simulate crash
❌ Crash simulated — workflow interrupted during step_2
```

Check what was saved before the crash:

In [19]:
graph.get_state(config)
# values:  {input: "start", step1: "done", step2: None, step3: None}
# next:    ("step_2",)   ← crashed here, still needs to run step_2


StateSnapshot(values={'input': 'start', 'step1': 'done'}, next=('step_2',), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f123979-42d0-63d5-8001-90ac31ab4e44'}}, metadata={'source': 'loop', 'step': 1, 'parents': {}}, created_at='2026-03-19T13:28:55.820782+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f123979-42b8-6626-8000-9e82c922c870'}}, tasks=(PregelTask(id='47101707-ae63-8b3c-7714-489b28c59976', name='step_2', path=('__pregel_pull', 'step_2'), error=None, interrupts=(), state=None, result=None),), interrupts=())

> 📌 *Even though the workflow crashed, step_1's output is already safely saved in the checkpointer. `next: ("step_2",)` tells us exactly where to resume.*

---

#### Step 2 — Resume from crash point

In [20]:
# pass None as input — signals "resume from last saved checkpoint"
# same thread_id — tells checkpointer which session to restore
final_state = graph.invoke(None, config=config)
print(final_state)

⏳ Step 2 running... interrupt now to simulate crash
✅ Step 3 executed
{'input': 'start', 'step1': 'done', 'step2': 'done', 'step3': 'done'}


```
⏳ Step 2 running...        ← resumes from step_2, NOT step_1
✅ Step 3 executed
{input: "start", step1: "done", step2: "done", step3: "done"}
```

> 📌 *`None` as the input is the resume signal — LangGraph fetches the last saved state for that `thread_id` and continues from the `next` node. Step 1 does NOT re-execute.*

---

#### Full state history after resume

In [21]:
list(graph.get_state_history(config))

[StateSnapshot(values={'input': 'start', 'step1': 'done', 'step2': 'done', 'step3': 'done'}, next=(), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f12397d-2d07-612c-8003-0c4aa89963f9'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}}, created_at='2026-03-19T13:30:40.910551+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f12397d-2cde-655d-8002-46d311e5bca0'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'input': 'start', 'step1': 'done', 'step2': 'done'}, next=('step_3',), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f12397d-2cde-655d-8002-46d311e5bca0'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-03-19T13:30:40.893863+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f123979-42d0-63d5-8001-90ac31ab4e44'}}, tasks=(PregelTask(id='8b604201-d244-c810

```
Checkpoint 5 — END:    {step1: done, step2: done, step3: done}  next: ()
Checkpoint 4 — step_3: {step1: done, step2: done, step3: None}  next: (step_3,)
Checkpoint 3 — step_2: {step1: done, step2: None,  step3: None} next: (step_2,)  ← crash point
Checkpoint 2 — step_1: {step1: None, step2: None,  step3: None} next: (step_1,)
Checkpoint 1 — START:  {}                                        next: (__start__,)
```

> 📌 *5 checkpoints instead of 4 — because the workflow ran in two separate invocations (first run up to crash, second run to completion). Both runs share the same `thread_id` so all checkpoints are grouped together.*

---

#### Why this matters in production

Without fault tolerance:
```
10-node workflow crashes at node 8 → restart from node 1 → waste time + API costs
```

With persistence:
```
10-node workflow crashes at node 8 → resume from node 8 → save time + API costs ✅
```

> 📌 *For long-running agentic workflows with expensive LLM calls, fault tolerance can save significant time and cost. This is one of LangGraph's key production advantages.*

---


---

### ⏳ Time Travel — Replay and Branch from Any Checkpoint

Persistence saves every intermediate state, which means you can go back to any checkpoint and re-execute the workflow from that point — with or without modifying the state first.

---

#### Replay from a past checkpoint

In [23]:
# step 1 — find the checkpoint you want to replay from
list(workflow.get_state_history(config1))
# → look for the snapshot where topic="pizza" but joke=None (just before generate_joke ran)

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why was the pizza maker always calm? Because he knew he could always "dough" it!', 'explanation': 'This joke plays on the wordplay between "dough" as in the dough used to make pizza crust and "do it." The pizza maker is always calm because he knows he can always "dough" it, referring to his ability to make pizza crust, but also implying that he can handle any situation that comes his way. The humor comes from the double meaning of "dough" in this context.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1238ba-251c-68ba-8002-65e178f9be54'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-03-19T12:03:25.589087+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1238ba-0f07-6ebc-8001-eef5c6626737'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'topic': 'pizza', 'joke': 'Why was the pizza maker always calm? Because he

In [24]:
# step 2 — fetch that specific checkpoint to confirm
workflow.get_state({"configurable": {
    "thread_id": "1",
    "checkpoint_id": "1f1238b9-f99c-6e5f-8000-87bd9a4245e1"
}})
# → values: {topic: "pizza", joke: None, explanation: None}
# → next:   ("generate_joke",)

StateSnapshot(values={'topic': 'pizza'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_id': '1f1238b9-f99c-6e5f-8000-87bd9a4245e1'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-03-19T12:03:21.027917+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1238b9-f993-67de-bfff-1a22a9007f22'}}, tasks=(PregelTask(id='5e37bd15-7884-f57e-eee9-409e8558deff', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result={'joke': 'Why was the pizza maker always calm? Because he knew he could always "dough" it!'}),), interrupts=())

In [25]:
# step 3 — invoke from that checkpoint (None = no new input, resume from here)
workflow.invoke(None, {"configurable": {
    "thread_id": "1",
    "checkpoint_id": "1f1238b9-f99c-6e5f-8000-87bd9a4245e1"
}})
# → generates a new joke on "pizza" (LLM is probabilistic — different output each time)
# → generates a new explanation for that joke

{'topic': 'pizza',
 'joke': 'Why did the pizza go to the party?\n\nBecause it wanted to get a "pizza" the action!',
 'explanation': 'This joke is a play on words. The phrase "get a piece of the action" means to be involved in an exciting or interesting situation. In this joke, the pizza went to the party because it wanted to be a part of the excitement or action happening there, but instead of saying "piece," it uses the word "pizza" to create a pun.'}

> 📌 *`checkpoint_id` in the config tells LangGraph exactly which snapshot to restore. Without it, `invoke(None, config)` resumes from the latest checkpoint. With it, you jump to any point in the execution history.*

---

#### What happens to state history after a replay?

In [26]:
list(workflow.get_state_history(config1))
# 4 original checkpoints  (first full run)
# + 2 new checkpoints     (time travel replay)
# = 6 total

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza go to the party?\n\nBecause it wanted to get a "pizza" the action!', 'explanation': 'This joke is a play on words. The phrase "get a piece of the action" means to be involved in an exciting or interesting situation. In this joke, the pizza went to the party because it wanted to be a part of the excitement or action happening there, but instead of saying "piece," it uses the word "pizza" to create a pun.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1239a5-542c-6bc7-8002-7c53b20c4e18'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-03-19T13:48:38.757238+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1239a5-3f13-66ae-8001-385611d8f43c'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza go to the party?\n\nBecause it wanted to get a "pizza" the actio

Each replay creates a new **branch** in the history — the original run is untouched.
```
Original run:    START → [pizza topic] → [pizza joke] → [pizza explanation]
                                  ↓
Time travel:             [pizza topic] → [new pizza joke] → [new explanation]
```

---

#### Update state before replaying — change the topic mid-history

In [27]:
# step 1 — update state at the checkpoint where topic="pizza"
#           this creates a new branch with topic="samosa"
workflow.update_state(
    {"configurable": {
        "thread_id": "1",
        "checkpoint_id": "1f1238b9-f99c-6e5f-8000-87bd9a4245e1",
        "checkpoint_ns": ""
    }},
    {"topic": "samosa"}   # override the state value at this point
)
# → creates a new checkpoint branching off from the pizza checkpoint

{'configurable': {'thread_id': '1',
  'checkpoint_ns': '',
  'checkpoint_id': '1f1239b8-ca02-6777-8001-183aa75092a1'}}

In [28]:
# step 2 — get the NEW checkpoint_id (the samosa branch, not the pizza one)
list(workflow.get_state_history(config1))
# → 7 checkpoints now — 4 original + 2 replay + 1 state update

[StateSnapshot(values={'topic': 'samosa'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1239b8-ca02-6777-8001-183aa75092a1'}}, metadata={'source': 'update', 'step': 1, 'parents': {}}, created_at='2026-03-19T13:57:21.140469+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1238b9-f99c-6e5f-8000-87bd9a4245e1'}}, tasks=(PregelTask(id='73c545ff-9d9c-ff8c-88c1-914b6e24a5ab', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza go to the party?\n\nBecause it wanted to get a "pizza" the action!', 'explanation': 'This joke is a play on words. The phrase "get a piece of the action" means to be involved in an exciting or interesting situation. In this joke, the pizza went to the party because it wanted to be a part of the excitement or ac

In [29]:
# step 3 — invoke from the NEW checkpoint (samosa branch)
workflow.invoke(None, {"configurable": {
    "thread_id": "1",
    "checkpoint_id": "1f1239b8-ca02-6777-8001-183aa75092a1"   # use the NEW id, not the pizza one
}})
# → joke is now about samosa ✅
# → explanation is for the samosa joke ✅

{'topic': 'samosa',
 'joke': 'Why did the samosa go to school? \nBecause it wanted to become a little more "well-rounded"!',
 'explanation': 'This joke plays on the double meaning of the phrase "well-rounded." In one context, being well-rounded means having a good variety of skills, knowledge, or experiences. In another context, a samosa is a triangular fried pastry that is traditionally filled with vegetables or meat. By saying that the samosa went to school to become more well-rounded, the joke is humorously suggesting that the samosa wants to change its shape from a triangle to a more rounded shape.'}

> ⚠️ *Common mistake: after `update_state`, invoking from the old `checkpoint_id` (pizza) instead of the new one (samosa) will still generate a pizza joke — because the old branch still points to pizza. Always get the new `checkpoint_id` after `update_state`.*

---

#### Full state history after all operations
```
Checkpoints 1-4 → original pizza run
Checkpoints 5-6 → time travel replay (pizza, different joke)
Checkpoint 7    → state update (topic changed to samosa)
Checkpoints 8-9 → time travel replay from samosa branch
```

---

#### When is time travel useful?

| Use case | How |
|---|---|
| **Debugging** | Replay a complex workflow from the exact node where something went wrong |
| **Experimentation** | Re-run an LLM node to get different outputs without restarting the whole workflow |
| **What-if analysis** | Change a state value mid-history and see how the rest of the workflow plays out differently |

> 📌 *Time travel is primarily a debugging and development tool. In production workflows you won't use it often — but for complex multi-node agentic systems, it's invaluable for tracing exactly what happened and why.*

---

### ✅ Persistence — All 4 Benefits

| Benefit | What it enables |
|---|---|
| **Short-term memory** | Chatbot remembers conversation history across `invoke()` calls |
| **Fault tolerance** | Resume workflow from crash point, not from scratch |
| **Human-in-the-Loop** | Pause execution, wait for human input, resume — powered by checkpoints |
| **Time travel** | Replay or branch from any past checkpoint for debugging and experimentation |

---
